In [ ]:
from yugiquery import *

init_notebook_mode(all_interactive=True)

header("Template")

---

Table of Contents <a class="jp-toc-ignore"></a>
=================
* [1 Data acquisition](#1-Data-acquisition)
  * [1.1 Fetch online data](#1.1-Fetch-online-data)
* [2 Check changes](#2-Check-changes)
  * [2.1 Load previous data](#2.1-Load-previous-data)
  * [2.2 Generate changelog](#2.2-Generate-changelog)
  * [2.3 Save data](#2.3-Save-data)
* [3 Data visualization](#3-Data-visualization)
  * [3.1 Full data](#3.1-Full-data)
  * [3.2 Your column](#3.2-Your-column)
    * [3.2.1 Number of unique entries](#3.2.1-Number-of-unique-entries)
    * [3.2.2 Bar plot](#3.2.2-Bar-plot)
    * [3.2.3 Crosstab](#3.2.3-Crosstab)
    * [3.2.4 Difference plots](#3.2.4-Difference-plots)
* [4 Epilogue](#4-Epilogue)
  * [4.1 HTML export](#4.1-HTML-export)
  <!-- * [4.2 Git](#4.2-Git) -->

# 1 Data acquisition

## 1.1 Fetch online data

In [ ]:
# Timestamp
timestamp = arrow.utcnow()

In [ ]:
# Fetch Monster
your_df = fetch_your_data()

# 2 Check changes

## 2.1 Load previous data

In [ ]:
# Get latest file if exist
previous_df, previous_ts = load_latest_data("your_data", return_ts=True)

if previous_df is not None:
    previous_df = previous_df.astype(your_df[previous_df.columns.intersection(your_df.columns)].dtypes.to_dict())
    print("File loaded")
else:
    print("No older files")

## 2.2 Generate changelog

In [ ]:
if previous_df is None:
    changelog = None
    print("Skipped")
else:
    changelog = generate_changelog(previous_df, your_df, col="Key_column")
    if not changelog.empty:
        display(changelog)
        changelog_path = dirs.DATA / make_filename(
            report="template",
            timestamp=timestamp,
            previous_timestamp=previous_ts,
        )
        changelog.to_csv(
            changelog_path,
            index=True,
        )
        _c_relpath = os.path.relpath(changelog_path, dirs.WORK)
        display(Markdown(f"Changelog saved to [{changelog_path.name}]({_c_relpath})"))

## 2.3 Save data

In [ ]:
if changelog is not None and changelog.empty:
    print("No changes. New data not saved")
else:
    data_path = dirs.DATA / make_filename(report="template", timestamp=timestamp)
    full_df.to_csv(
        data_path,
        index=False,
    )
    _d_relpath = os.path.relpath(data_path, dirs.WORK)
    display(Markdown(f"Data saved to [{data_path.name}]({_d_relpath})"))

# 3 Data visualization

## 3.1 Full data

In [ ]:
your_df

Full data available to download [here](plot.colors_dictdata)

## 3.2 Your column

In [ ]:
print("Total number of Your_column:", your_df["Your_column"].nunique())

### 3.2.1 Number of unique entries

In [ ]:
your_df.drop(columns=["unnecessary_columns"]).groupby("Your_column").nunique()

### 3.2.2 Bar plot

In [ ]:
your_colors = [plot.colors_dict[i] for i in your_df["Your_column"].value_counts().index]
your_df["Your_column"].value_counts().plot.bar(
    figsize=(18, 6), grid=True, rot=0, color=card_type_colors, title="Your_column"
)
plt.show()

### 3.2.3 Crosstab

In [ ]:
your_crosstab = pd.crosstab(your_df["Your_column"], your_df["Other_column"])
your_crosstab

In [ ]:
plt.figure(figsize=(16, 10))
sns.heatmap(your_crosstab, annot=True, fmt="g", cmap="viridis", square=True)
plt.show()

### 3.2.4 Difference plots

In [ ]:
diff_colors = {
    "sel_1": plot.colors_dict["sel_1"],
    "sel_2": plot.colors_dict["sel_2"],
}
diff.plot.bar(figsize=(18, 6), stacked=True, grid=True, rot=45, color=diff_colors)
plt.show()

# 4 Epilogue

In [ ]:
benchmark(report="Template", timestamp=timestamp)

In [ ]:
footer(timestamp)

## 4.1 HTML export

In [ ]:
# Save notebook on disck before generating HTML report
save_notebook()

In [ ]:
export_notebook(dirs.NOTEBOOKS.user / "Template.ipynb")

## 4.2 Git

In [ ]:
print(git.commit("*[Tt]emplate", f"Your update - {timestamp.isoformat()}"))